# S08 — one of six independent dense workers
Open this notebook in six CUDA sessions with WORKER_ID 0–5. Every session uses the matching worker ZIP from one shared immutable plan. Each worker has its own local and Drive output. Never launch two writers for the same worker. No distributed training/NCCL is used. See docs/RETRIEVAL_STAGES7_8.md. All tests/jobs are owner-run; QA/calibration/comparisons remain deferred.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
WORKER_ID = 0  # unique in each session: 0, 1, 2, 3, 4, 5
DRIVE_RUN = Path('/content/drive/MyDrive/ViBioMIR/dense_stage8_bge_structure')
BUNDLE_DIR = DRIVE_RUN / 'bundles'
BUNDLE = BUNDLE_DIR / f'worker_{WORKER_ID}.zip'
PROJECT = Path('/content') / f'vibiomir_dense_worker_{WORKER_ID}'
RUN_DIR = Path('/content') / f'dense_worker_{WORKER_ID}'
BACKUP_DIR = DRIVE_RUN / f'worker_{WORKER_ID}'
BATCH_SIZE = 16  # T4 start; L4:32, A100:64 after pilot
PADDED_TOKENS = 8192  # L4:16384, A100:32768 after pilot
WAVE_CHUNKS = 128
STOP_AFTER_BATCHES = 1  # then None; rerun the encode cell
assert BUNDLE.is_file(), 'Upload this worker ZIP and edit DRIVE_RUN first.'
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')


In [ ]:
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    meta = json.loads(archive.read('BUNDLE.json'))
    assert meta['worker_id'] == WORKER_ID, 'Wrong worker ZIP'
    required = sum(info.file_size for info in archive.infolist())
    assert shutil.disk_usage('/content').free > required * 3 + 8 * 1024**3, 'Need local space for model, input and worker output.'
    for info in archive.infolist():
        assert PROJECT.resolve() in (PROJECT / info.filename).resolve().parents, 'Unsafe ZIP path'
        assert (info.external_attr >> 16) & 0o170000 != 0o120000, 'Symlink ZIP entry'
    archive.extractall(PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-stage8-colab.txt'], check=True)
print('If a runtime restart is requested, restart and rerun setup.')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime; TPU is not supported by this worker.'
plan = json.loads((PROJECT / 'plan.json').read_text())
assert plan['signature'] == meta['plan_signature']
print({'gpu': torch.cuda.get_device_name(0), 'worker': plan['workers'][WORKER_ID], 'encoding':plan['encoding']})
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_stage78*.py', '-v'], check=True)


In [ ]:
command = [sys.executable, '-m', 'vibiomir.dense_stage8', 'encode-worker',
    '--plan', str(PROJECT / 'plan.json'), '--chunk-dir', str(PROJECT / 'chunks'),
    '--tokenizer-dir', str(PROJECT / 'tokenizer'), '--output-dir', str(RUN_DIR),
    '--worker-id', str(WORKER_ID), '--backup-dir', str(BACKUP_DIR),
    '--batch-size', str(BATCH_SIZE), '--padded-tokens', str(PADDED_TOKENS), '--wave-chunks', str(WAVE_CHUNKS)]
if STOP_AFTER_BATCHES is not None:
    command += ['--stop-after-batches', str(STOP_AFTER_BATCHES)]
subprocess.run(command, check=True)
state = json.loads((RUN_DIR / 'manifest.json').read_text())
print('Worker state:', state['state'])
if state['state'] != 'complete':
    print('Set STOP_AFTER_BATCHES=None and rerun this cell. Committed batches are reused; Drive restore is automatic after reset.')


In [ ]:
state = json.loads((RUN_DIR / 'manifest.json').read_text())
if state['state'] == 'complete':
    subprocess.run([sys.executable, '-m', 'vibiomir.dense_stage8', 'validate-worker',
        '--plan', str(PROJECT / 'plan.json'), '--chunk-dir', str(PROJECT / 'chunks'),
        '--output-dir', str(RUN_DIR), '--worker-id', str(WORKER_ID)], check=True)
    shutil.copy2(RUN_DIR / 'VALIDATION.json', BACKUP_DIR / 'VALIDATION.json')
    print('Complete worker persisted:', BACKUP_DIR)
else:
    print('Finish all assigned batches before validation.')


In [ ]:
ENCODE_QUERIES = WORKER_ID == 0
query_files = [p for p in (PROJECT / 'queries.parquet', PROJECT / 'queries.json') if p.is_file()]
if ENCODE_QUERIES and query_files:
    QUERY_DIR = Path('/content/dense_query_vectors')
    subprocess.run([sys.executable, '-m', 'vibiomir.dense_stage8', 'encode-queries',
        '--plan', str(PROJECT / 'plan.json'), '--tokenizer-dir', str(PROJECT / 'tokenizer'),
        '--queries', str(query_files[0]), '--output-dir', str(QUERY_DIR), '--batch-size', str(BATCH_SIZE)], check=True)
    shutil.copytree(QUERY_DIR, DRIVE_RUN / 'query_vectors', dirs_exist_ok=True)
    print('Query vectors copied to Drive. No relevance evaluation performed.')


## Optional central merge on one session
Run only after all six worker manifests are complete. This is CPU/disk work; the other five GPUs can stop. Local merge is also documented. Source chunks from all ZIPs are needed to validate every mapping. Merged output copies vector batches, so both worker copies and merged data need local disk. Final comparisons and ANN recall measurement are deferred.

In [ ]:
MERGE_ALL = False  # enable on one session only, once all workers finish
if MERGE_ALL:
    assert len(plan['workers']) == 6
    expected_bytes = plan['source']['chunks'] * plan['encoding']['model_spec']['dimension'] * 2
    assert shutil.disk_usage('/content').free > expected_bytes * 3 + 10 * 1024**3, 'Merge locally instead if Colab disk is insufficient.'
    for worker in plan['workers']:
        source = DRIVE_RUN / f"worker_{worker['id']}"
        assert (source / 'manifest.json').is_file(), f'Missing worker: {source}'
        manifest = json.loads((source / 'manifest.json').read_text())
        assert manifest['state'] == 'complete' and manifest['identity']['plan_signature'] == plan['signature'], 'Incomplete/mismatched worker'
    MERGE_WORKERS = Path('/content/dense_merge_workers')
    MERGED = Path('/content/dense_merged')
    for worker in plan['workers']:
        with zipfile.ZipFile(BUNDLE_DIR / f"worker_{worker['id']}.zip") as archive:
            assert json.loads(archive.read('plan.json')) == plan, 'Mixed plans'
            for info in archive.infolist():
                if info.filename.startswith('chunks/'):
                    assert PROJECT.resolve() in (PROJECT / info.filename).resolve().parents
                    archive.extract(info, PROJECT)
        print('Restoring worker', worker['id'], flush=True)
        shutil.copytree(DRIVE_RUN / f"worker_{worker['id']}", MERGE_WORKERS / f"worker_{worker['id']}", dirs_exist_ok=True)
    subprocess.run([sys.executable, '-m', 'vibiomir.dense_stage8', 'merge',
        '--plan', str(PROJECT / 'plan.json'), '--chunk-dir', str(PROJECT / 'chunks'),
        '--workers-dir', str(MERGE_WORKERS), '--output-dir', str(MERGED)], check=True)
    shutil.copytree(MERGED, DRIVE_RUN / 'merged', dirs_exist_ok=True)
    print('Merged vectors/mapping/validation persisted:', DRIVE_RUN / 'merged')
